# 📊 Notebook 01: Làm sạch dữ liệu & Phân tích khám phá (Data Cleaning & EDA)
**Dự án:** Phân tích quan điểm sinh viên CMC (`cmc-sentiment-analysis`)  
**Nguồn dữ liệu:** Toàn bộ 3 bộ dữ liệu JSON trong thư mục `data/` (`dataset_*.json`):
1. `data/dataset_facebook-groups-scraper_2026-11-08_10-57-09-245.json`
2. `data/dataset_facebook-groups-scraper_2026-11-08_11-44-31-370.json`
3. `data/dataset_facebook-posts-scraper_2026-11-08_12-08-06-404.json`

---

## 🧩 Cell 1: Đọc toàn bộ các file JSON `data/dataset_*.json`, bóc tách bài viết chính (`text`) lẫn mảng bình luận (`comments`) thành DataFrame
- Tự động quét tất cả các file `dataset_*.json` trong thư mục `data/`.
- Trích xuất nội dung bài viết chính (`type = 'post'`, bao gồm cả nội dung `sharedPost` nếu có) và đệ quy lấy toàn bộ bình luận trong `comments` / `topComments` (`type = 'comment'`).
- Đưa về cấu trúc `pandas.DataFrame` gồm các cột: `id`, `type`, `time`, `content`, `likes`, `label`.

In [1]:
import json
import re
import unicodedata
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Tự động xác định thư mục gốc của dự án (hỗ trợ chạy từ cả thư mục gốc lẫn thư mục notebooks/)
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data"
TEENCODE_PATH = DATA_DIR / "teencode.json"
OUTPUT_CSV_PATH = DATA_DIR / "cmc_dataset_clean.csv"

def parse_likes(val) -> int:
    """Chuyển đổi lượt thích về kiểu số nguyên an toàn."""
    if val is None:
        return 0
    try:
        return int(str(val).replace(",", "").strip())
    except (ValueError, TypeError):
        return 0

def extract_all_comments(post: dict) -> list:
    """Lấy toàn bộ bình luận từ `comments` và `topComments` (kể cả trong `sharedPost` và phản hồi lồng nhau)."""
    queue = []
    for key in ("comments", "topComments"):
        items = post.get(key)
        if isinstance(items, list):
            queue.extend(items)

    shared = post.get("sharedPost")
    if isinstance(shared, dict):
        for key in ("comments", "topComments"):
            items = shared.get(key)
            if isinstance(items, list):
                queue.extend(items)

    collected = []
    while queue:
        c = queue.pop(0)
        if not isinstance(c, dict):
            continue
        collected.append(c)
        for reply_key in ("comments", "replies", "topComments"):
            replies = c.get(reply_key)
            if isinstance(replies, list):
                queue.extend(replies)
    return collected

# Đọc tất cả các file dataset_*.json trong thư mục data/
json_files = sorted(DATA_DIR.glob("dataset_*.json"))
print(f"Đã đọc {len(json_files)} file JSON trong thư mục data/:")

records = []
for json_path in json_files:
    with open(json_path, mode="r", encoding="utf-8") as f:
        raw_posts = json.load(f)

    file_post_cnt = 0
    file_comm_cnt = 0
    for post_idx, post in enumerate(raw_posts, start=1):
        if not isinstance(post, dict):
            continue

        # 1. Bài đăng chính (post)
        post_text = post.get("text") or post.get("message") or ""
        shared = post.get("sharedPost")
        if isinstance(shared, dict):
            shared_text = shared.get("text") or shared.get("message") or ""
            if shared_text and shared_text not in post_text:
                post_text = f"{post_text} {shared_text}".strip() if post_text else shared_text

        records.append({
            "id": str(post.get("legacyId") or post.get("postId") or post.get("id") or f"post_{json_path.stem}_{post_idx}"),
            "type": "post",
            "time": str(post.get("time") or post.get("date") or ""),
            "content": post_text,
            "likes": parse_likes(post.get("likesCount", post.get("likes", post.get("reactionLikeCount", 0)))),
            "label": ""
        })
        file_post_cnt += 1

        # 2. Mảng bình luận (comments)
        for c_idx, comment in enumerate(extract_all_comments(post), start=1):
            records.append({
                "id": str(comment.get("commentId") or comment.get("id") or f"comment_{json_path.stem}_{post_idx}_{c_idx}"),
                "type": "comment",
                "time": str(comment.get("date") or comment.get("time") or post.get("time") or ""),
                "content": comment.get("text") or comment.get("message") or "",
                "likes": parse_likes(comment.get("likesCount", comment.get("likes", comment.get("reactionLikeCount", 0)))),
                "label": ""
            })
            file_comm_cnt += 1
    print(f"- {json_path.name}: {file_post_cnt:,} posts, {file_comm_cnt:,} comments")

df_raw = pd.DataFrame(records)
print(f"\nTổng số bản ghi thô trích xuất được: {len(df_raw):,} dòng")
print(df_raw["type"].value_counts())

Đã đọc 3 file JSON trong thư mục data/:
- dataset_facebook-groups-scraper_2026-11-08_10-57-09-245.json: 500 posts, 696 comments
- dataset_facebook-groups-scraper_2026-11-08_11-44-31-370.json: 248 posts, 244 comments
- dataset_facebook-posts-scraper_2026-11-08_12-08-06-404.json: 751 posts, 324 comments

Tổng số bản ghi thô trích xuất được: 2,763 dòng
type
post       1499
comment    1264
Name: count, dtype: int64


## 🧹 Cell 2: Lọc bỏ dòng trống, dòng trùng lặp, câu dưới 3 từ và hiển thị 10 dòng đầu tiên
- Chuẩn hóa khoảng trắng và ký tự xuống dòng trong cột `content`.
- Loại bỏ các dòng rỗng (`content == ""`).
- Loại bỏ các câu quá ngắn có **số từ < 3**.
- Loại bỏ các nội dung bị trùng lặp giữa các lần cào (không phân biệt chữ hoa/thường).
- Hiển thị 10 dòng đầu tiên bằng `df.head(10)`.

In [2]:
df = df_raw.copy()

# 1. Chuẩn hóa khoảng trắng và bỏ dòng trống
df["content"] = df["content"].fillna("").astype(str).apply(lambda s: re.sub(r"\s+", " ", s).strip())
df = df[df["content"] != ""].copy()

# 2. Tính số lượng từ và lọc bỏ câu dưới 3 từ
df["word_count"] = df["content"].apply(lambda s: len(s.split()))
df = df[df["word_count"] >= 3].copy()

# 3. Loại bỏ các dòng có nội dung trùng lặp (không phân biệt hoa/thường)
df["content_lower"] = df["content"].str.lower()
df = df.drop_duplicates(subset=["content_lower"], keep="first").drop(columns=["content_lower"]).reset_index(drop=True)

print(f"Số dòng còn lại sau khi làm sạch: {len(df):,} dòng (Đã loại bỏ {len(df_raw) - len(df):,} dòng trống/ngắn/trùng lặp)")
df.head(10)

Số dòng còn lại sau khi làm sạch: 2,514 dòng (Đã loại bỏ 249 dòng trống/ngắn/trùng lặp)


,id,type,time,content,likes,label,word_count
0,1424542162993445,post,2026-11-07T13:46:32.000Z,Ai đi 10fest không ^^,5,,5
1,1424702252977436,comment,2026-11-07T16:46:34.000Z,FPT out thì xem gì nữa :(,0,,7
2,1424851169629211,comment,2026-11-07T19:57:51.000Z,tôi đeo sẵn vòng tay rồi chờ day 2😋,1,,9
3,1424192703028391,post,2026-11-07T05:45:00.000Z,"🔥THÔNG BÁO : GIA HẠN ĐƠN TUYỂN THÀNH VIÊN GEN 4🔥 Trong thời gian mở đơn tuyển thành viên Gen 4 vừa qua, THE IX đã nhận đ...",8,,638
4,1424384059675922,comment,2026-11-07T10:41:20.000Z,Yêu cầu các bạn casting mỗi bạn nhảy 10p,0,,9
5,1424680239646304,comment,2026-11-07T16:17:52.000Z,cho em vô với,0,,4
6,1423756656405329,post,2026-11-06T18:06:38.000Z,"miễn phí ship kvuc hà đông ,bà đình và anh em đại học cmc.Ae lên!",8,,15
7,1423783169736011,comment,2026-11-06T18:44:49.000Z,cho tôi một butter nhé,1,,5
8,1423758509738477,comment,2026-11-06T18:08:55.000Z,mọi người qtam nhắn tin vợ tôi Nguyễn Diệu Linh,0,,10
9,1423586416422353,post,2026-11-06T14:43:56.000Z,Trường mình có những câu lạc bộ nào vậy anh chị. em cần tìm 1 câu lạc bộ ưng ý á. Trừ ca hát nhảy múa ra nhé🙃,1,,28


## 📈 Cell 3: Thống kê số lượng bài viết vs bình luận, độ dài trung bình của câu và vẽ biểu đồ trực quan
- Thống kê số lượng và tỷ lệ phần trăm giữa **Bài đăng (`post`)** và **Bình luận (`comment`)** trên tổng 3 bộ dữ liệu.
- Tính độ dài trung bình của câu (theo số từ và số ký tự) cho từng nhóm.
- Vẽ biểu đồ trực quan gồm:
  1. Biểu đồ cột so sánh số lượng Bài đăng (`post`) vs Bình luận (`comment`).
  2. Biểu đồ phân phối độ dài câu (số từ trên mỗi câu).

In [3]:
# Tính thêm độ dài ký tự
df["char_count"] = df["content"].str.len()

# Bảng thống kê tổng hợp theo loại (post vs comment)
type_counts = df["type"].value_counts()
avg_words_overall = df["word_count"].mean()
avg_chars_overall = df["char_count"].mean()

print("=== THỐNG KÊ TỔNG QUAN DỮ LIỆU (3 DATASETS) ===")
print(f"- Tổng số bài đăng (post)      : {type_counts.get('post', 0):,} bài ({type_counts.get('post', 0)/len(df)*100:.1f}%)")
print(f"- Tổng số bình luận (comment)  : {type_counts.get('comment', 0):,} bình luận ({type_counts.get('comment', 0)/len(df)*100:.1f}%)")
print(f"- Tổng cộng                    : {len(df):,} dòng")
print(f"- Độ dài câu trung bình (từ)   : {avg_words_overall:.2f} từ/câu")
print(f"- Độ dài câu trung bình (ký tự): {avg_chars_overall:.2f} ký tự/câu\n")

summary_stats = df.groupby("type").agg(
    so_luong=("id", "count"),
    so_tu_tb=("word_count", "mean"),
    so_tu_median=("word_count", "median"),
    so_ky_tu_tb=("char_count", "mean"),
    luot_thich_tb=("likes", "mean")
).round(2)
display(summary_stats)

# Vẽ biểu đồ trực quan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Số lượng Bài viết vs Bình luận
colors = ["#3B82F6", "#10B981"]
bars = axes[0].bar(type_counts.index, type_counts.values, color=colors, width=0.5, edgecolor="black")
axes[0].set_title("Số lượng Bài viết (post) vs Bình luận (comment)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Loại dữ liệu (type)")
axes[0].set_ylabel("Số lượng bản ghi")
axes[0].grid(axis="y", linestyle="--", alpha=0.5)
for bar in bars:
    height = bar.get_height()
    axes[0].annotate(
        f"{int(height):,}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        fontweight="bold"
    )

# Biểu đồ 2: Phân phối độ dài câu (giới hạn hiển thị <= 200 từ để dễ quan sát)
clipped_words = df["word_count"].clip(upper=200)
axes[1].hist(clipped_words, bins=30, color="#6366F1", edgecolor="white", alpha=0.85)
axes[1].axvline(avg_words_overall, color="#EF4444", linestyle="--", linewidth=2, label=f"Trung bình: {avg_words_overall:.1f} từ")
axes[1].set_title("Phân phối độ dài câu (Số từ / bản ghi)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Số từ (giới hạn hiển thị tối đa 200 từ)")
axes[1].set_ylabel("Tần suất")
axes[1].legend()
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

=== THỐNG KÊ TỔNG QUAN DỮ LIỆU (3 DATASETS) ===
- Tổng số bài đăng (post)      : 1,431 bài (56.9%)
- Tổng số bình luận (comment)  : 1,083 bình luận (43.1%)
- Tổng cộng                    : 2,514 dòng
- Độ dài câu trung bình (từ)   : 136.34 từ/câu
- Độ dài câu trung bình (ký tự): 654.07 ký tự/câu


,type,so_luong,so_tu_tb,so_tu_median,so_ky_tu_tb,luot_thich_tb
0,comment,1083,11.17,7.0,47.71,1.47
1,post,1431,231.08,137.0,1112.97,49.29


## 🔤 Cell 4: Chuẩn hóa từ viết tắt qua `data/teencode.json` và xuất ra file `data/cmc_dataset_clean.csv`
- Tải từ điển chuẩn hóa teencode/viết tắt sinh viên từ `data/teencode.json`.
- Ánh xạ và thay thế các từ viết tắt độc lập trong câu.
- Lưu bảng dữ liệu sạch gồm 6 cột chuẩn (`id`, `type`, `time`, `content`, `likes`, `label`) ra file `data/cmc_dataset_clean.csv` với bảng mã `utf-8-sig`.

In [4]:
# Đọc từ điển teencode
with open(TEENCODE_PATH, mode="r", encoding="utf-8") as f:
    teencode_raw = json.load(f)

teencode_dict = {str(k).strip().lower(): str(v).strip() for k, v in teencode_raw.items() if str(k).strip()}
print(f"Đã nạp {len(teencode_dict)} mục từ điển teencode từ: {TEENCODE_PATH.name}")

# Biên dịch biểu thức chính quy để thay thế các từ viết tắt đứng độc lập theo ranh giới từ
sorted_keys = sorted(teencode_dict.keys(), key=len, reverse=True)
teencode_pattern = re.compile(
    r"(?<!\w)(" + "|".join(re.escape(k) for k in sorted_keys) + r")(?!\w)",
    flags=re.IGNORECASE
)

def normalize_teencode_text(text: str) -> str:
    """Chuẩn hóa Unicode NFC và thay thế từ viết tắt sinh viên bằng từ điển teencode.json."""
    text = unicodedata.normalize("NFC", str(text))
    def _replace(match: re.Match) -> str:
        word = match.group(0)
        replacement = teencode_dict.get(word.lower(), word)
        return replacement
    normalized = teencode_pattern.sub(_replace, text)
    return re.sub(r"\s+", " ", normalized).strip()

# Áp dụng chuẩn hóa teencode lên cột content
df["content"] = df["content"].apply(normalize_teencode_text)

# Chọn đúng 6 cột chuẩn và xuất ra CSV với encoding utf-8-sig
export_columns = ["id", "type", "time", "content", "likes", "label"]
df_export = df[export_columns].copy()
df_export.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"\n✅ Đã xuất thành công {len(df_export):,} dòng ra file: {OUTPUT_CSV_PATH}")
df_export.head(10)

Đã nạp 119 mục từ điển teencode từ: teencode.json

✅ Đã xuất thành công 2,514 dòng ra file: c:\Users\viloc\cmc-sentiment-analysis\data\cmc_dataset_clean.csv


,id,type,time,content,likes,label
0,1424542162993445,post,2026-11-07T13:46:32.000Z,Ai đi 10fest không ^^,5,
1,1424702252977436,comment,2026-11-07T16:46:34.000Z,FPT out thì xem gì nữa :(,0,
2,1424851169629211,comment,2026-11-07T19:57:51.000Z,tôi đeo sẵn vòng tay rồi chờ day 2😋,1,
3,1424192703028391,post,2026-11-07T05:45:00.000Z,"🔥THÔNG BÁO : GIA HẠN ĐƠN TUYỂN THÀNH VIÊN GEN 4🔥 Trong thời gian mở đơn tuyển thành viên Gen 4 vừa qua, THE IX đã nhận đ...",8,
4,1424384059675922,comment,2026-11-07T10:41:20.000Z,Yêu cầu các bạn casting mỗi bạn nhảy 10p,0,
5,1424680239646304,comment,2026-11-07T16:17:52.000Z,cho em vô với,0,
6,1423756656405329,post,2026-11-06T18:06:38.000Z,"miễn phí ship kvuc hà đông ,bà đình và anh em đại học cmc.Ae lên!",8,
7,1423783169736011,comment,2026-11-06T18:44:49.000Z,cho tôi một butter nhé,1,
8,1423758509738477,comment,2026-11-06T18:08:55.000Z,mọi người qtam nhắn tin vợ tôi Nguyễn Diệu Linh,0,
9,1423586416422353,post,2026-11-06T14:43:56.000Z,Trường mình có những câu lạc bộ nào vậy anh chị. em cần tìm 1 câu lạc bộ ưng ý á. Trừ ca hát nhảy múa ra nhé🙃,1,
